# ប្រព័ន្ធ POS Django — ផ្នែក 2: Forms, Authentication & Deployment 🛒

## មាតិកា (Table of Contents)
1. [Learning Outcomes](#learning-outcomes)
2. [Quick Recap: What We Built in Part 1](#recap)
3. [Django Forms — Create Orders via the Web](#django-forms)
4. [User Authentication — Staff Login & Logout](#user-authentication)
5. [Protecting Views with `@login_required`](#protecting-views)
6. [Tying Orders to the Logged-In Cashier](#tying-orders)
7. [Deployment Basics](#deployment)
8. [Homework](#homework)

> **គោលបំណងគម្រោង (ផ្នែក 2)**: ពង្រីកមូលដ្ឋាន POS ពី​ផ្នែក 1 — បន្ថែមសំណុំតាមវេបសម្រាប់ cashier ដើម្បីបញ្ចូល order, អោយបុគ្គលិកចូលជាមុនពេលប្រើប្រព័ន្ធ, និងរៀបចំ app សម្រាប់ម៉ាស៊ីនម៉ាស៊ែវើ។


---

## លទ្ធផលការសិក្សា (Learning Outcomes)

នៅចុងបញ្ចប់មេរៀននេះ អ្នកអាចធ្វើបាន៖

| # | លទ្ធផល |
|---|---|
| 1 | **សាង Django Forms** — បង្កើត `ModelForm` ជា class ដើម្បីទទួល និងផ្ទៀងផ្ទាត់ទិន្នន័យពី browser |
| 2 | **ដោះស្រាយ POST Requests** — សរសេរ view ដែលដំណើរការ form submission និងរក្សាទុកទៅ database យ៉ាងសុវត្ថិភាព |
| 3 | **អនុវត្ត Authentication** — ប្រើប្រព័ន្ធ login/logout និង password ដែល Django ផ្តល់ |
| 4 | **ការពារ Views** — កំណត់ឲ្យទំព័រមួយៗសម្រាប់បុគ្គលិកដែល login តែប៉ុណ្ណោះ ដោយប្រើ `@login_required` |
| 5 | **ភ្ជាប់ទិន្នន័យទៅឲ្យ User** — ជំនួស `cashier` ជា plain-text ដោយ ForeignKey ទៅ `User` |
| 6 | **យល់ពី Deployment** — រៀបចំ Django project សម្រាប់ម៉ាស៊ីនផលិត (DEBUG off, `SECRET_KEY`, static files, `gunicorn`) |

### ជំនាញដែលអ្នកនឹងអនុវត្ត
- ធ្វើការជាមួយ Django **CSRF protection** (ការការ៉េប្រឆាំង cross-site form attack)
- ប្រើ **`request.user`** ដើម្បីកំណត់អ្នកដែលបាន login
- សរសេរ **`is_valid()` / `cleaned_data`** សម្រាប់ form validation
- អាន app ផ្ទាល់ដែល Django ផ្តល់ **`django.contrib.auth`**
- រៀបចំ `settings.py` សម្រាប់ production ដោយប្រើ environment variables

---


---

## សង្ខេបឆាប់ៗ: អ្វីដែលយើងបានបង្កើតនៅ ផ្នែក 1

នៅផ្នែក 1 យើងបានរៀបចំ skeleton នៃប្រព័ន្ធ POS៖

```
posdb/
├── manage.py
├── posdb/
│   ├── settings.py      ← registered 'sales' app
│   └── urls.py          ← routes /sales/ to sales/urls.py
└── sales/
    ├── models.py        ← Product, Order, OrderItem
    ├── views.py         ← product_list(), product_detail(), order_list()
    ├── urls.py          ← URL patterns
    ├── admin.py         ← admin panel configuration
    └── templates/sales/
        ├── product_list.html
        └── order_list.html
```

**អ្វីខ្វះនៅផ្នែក 1?**
- Staff អាចបង្កើត orders តែតាម admin panel តែប៉ុណ្ណោះ — មិនមាន web page ធម្មតា
- អ្នកទស្សនាណាមួយអាចចូល `/sales/orders/` បាន — មិនមានការទាមទារ login
- ប្រដាប់ `cashier` គឺជា free text មិនភ្ជាប់ទៅ account ពិតប្រាកដ
- គម្រោងមិនទាន់រួចសម្រាប់ដាក់លើម៉ាស៊ីនពិត

**ផ្នែក 2 ជួយដោះស្រាយរឿងទាំងនេះ។**

---


---

## Django Forms — បង្កើត Orders តាម Web

### Django Form ជាអ្វី?

**Django Form** គឺជា class Python ដែល៖
- កំណត់វាលដែលបង្ហាញក្នុង HTML `<form>`
- ផ្ទៀងផ្ទាត់ទិន្នន័យដែលអ្នកប្រើបញ្ចូល
- បម្លែង POST raw data ទៅជា Python object ស្អាត

**`ModelForm`** ជា shortcut ដែលបង្កើត form តាម Model ដោយផ្ទាល់ ដើម្បីកុំឲ្យចម្លងកូដ

```
Cashier clicks "＋ New Order"
        ↓
create_order view instantly creates an Order in the database
        ↓
Redirects to add_item page → cashier scans/selects products
        ↓
Each product form submission → POST request → form.is_valid() → item saved
```

> **សេចក្ដីសម្រេចលើការរចនា**: ជំនួសចង្អុលទៅទំព័រផ្សេងទៀត សម្រាប់ 
 ការ​ចុច **＋ New Order** នឹងបង្កើត open order ខ្លាំងៗ ហើយបញ្ជូន cashier ទៅផ្ទៃដាក់ item មួយភ្លាម។ នេះកាត់បន្ថយកន្លែងចុចមួយ និងសម្បូរពី UX របស់ till ផ្ទាល់។

### ដំណាក់កាល 1 — បង្កើត `sales/forms.py`

យើងត្រូវការ `OrderItemForm` តែមួយ — `OrderForm` មិនចាំបាច់ ព្រោះ order ត្រូវបានបង្កើតដោយស្វ័យប្រវត្តិ៖


In [ ]:
# sales/forms.py

from django import forms
from .models import OrderItem


class OrderItemForm(forms.ModelForm):
    """One line item to add to an order."""
    class Meta:
        model  = OrderItem
        fields = ['product', 'quantity']

    def clean_quantity(self):
        """Validation: quantity must be at least 1."""
        qty = self.cleaned_data['quantity']
        if qty < 1:
            raise forms.ValidationError("Quantity must be at least 1.")
        return qty


### គំនិតសម្រាប់ Form សំខាន់ៗ

| Concept | តើវាធ្វើអ្វី |
|---|---|
| `ModelForm` | បង្កើត field ពី Model ដោយស្វ័យ — មិនចាំបាច់ចម្លង |
| `fields = [...]` | បង្ហាញ field ទាំងនេះក្នុង form តែប៉ុណ្ណោះ |
| `form.is_valid()` | ប្រតិបត្តិការផ្ទៀងផ្ទាត់ទាំងអស់; បង្វិល `True` ប្រសិនបើទិន្នន័យស្អាត |
| `form.cleaned_data` | dictionary របស់តម្លៃ Python ដែលបាន validated |
| `clean_<field>()` | បន្ថែម logic validation សម្រាប់ field ផ្សេងៗ |
| `form.save()` | បង្កើត (ឬ update) instance នៅក្នុង database |
| `commit=False` | បង្កើត object មិនទាន់រក្សាទុក — អនុញ្ញាតកំណត់ field បន្ថែមមុន |

### ដំណាក់កាល 2 — ធ្វើបន្ទាន់ `views.py` ដើម្បីដំណើរការ Form

មាន view ច្រេីនពីរដែលគួរផ្នែក order workflow៖
- `create_order` — បង្កើត open order ហើយ redirect តែម្តង (មិនបង្ហាញ form)
- `add_item` — ឲ្យ cashier បញ្ចូល product lines ហើយ mark order ជា paid


In [ ]:
# sales/views.py  (add these imports and views at the bottom of the file)

from django.shortcuts import render, get_object_or_404, redirect
from django.contrib.auth.decorators import login_required
from .models import Product, Order, OrderItem
from .forms import OrderItemForm


# ── existing views (product_list, product_detail, order_list) stay above ──


@login_required
def create_order(request):
    """Instantly create an open order and jump straight to the add-items page."""
    order = Order.objects.create(
        cashier=request.user,
        status='open',
    )
    return redirect('add_item', pk=order.pk)


@login_required
def add_item(request, pk):
    """
    Let the cashier add line items to an open order, then mark it paid.
    """
    order = get_object_or_404(Order, pk=pk)

    if request.method == 'POST':
        # "Mark as Paid" button
        if 'mark_paid' in request.POST:
            order.status = 'paid'
            order.save()
            return redirect('order_list')

        # Add a line item
        item_form = OrderItemForm(request.POST)
        if item_form.is_valid():
            item = item_form.save(commit=False)
            item.order      = order
            item.unit_price = item.product.price   # snapshot the current price
            item.save()
            return redirect('add_item', pk=order.pk)
    else:
        item_form = OrderItemForm()

    return render(request, 'sales/add_item.html', {
        'order':     order,
        'item_form': item_form,
        'items':     order.items.select_related('product'),
    })


### ដំណាក់កាល 3 — បង្កើត Template សម្រាប់ Add Item

តែ **មួយ** template គែមគគ្រប់ប្រតិបត្តិការ order workflow ឥឡូវនេះ។ បង្កើត `sales/templates/sales/add_item.html`:


In [ ]:
<!-- sales/templates/sales/add_item.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Order #{{ order.pk }}</title>
    <style>
        body  { font-family: Arial, sans-serif; max-width: 860px; margin: 40px auto; background: #f0f4f8; padding: 0 20px; }
        table { width: 100%; border-collapse: collapse; background: white; border-radius: 10px;
                overflow: hidden; box-shadow: 0 2px 6px rgba(0,0,0,0.08); margin-bottom: 24px; }
        th { background: #2b6cb0; color: white; padding: 10px 14px; text-align: left; }
        td { padding: 10px 14px; border-bottom: 1px solid #e2e8f0; }
        .total { font-size: 1.3em; font-weight: bold; color: #276749; }
        form.add  { background: white; padding: 20px 24px; border-radius: 10px;
                    box-shadow: 0 2px 6px rgba(0,0,0,0.08); display: flex; gap: 12px; align-items: flex-end; }
        form.add label { font-weight: bold; display: block; margin-bottom: 4px; }
        form.add select, form.add input { padding: 8px 10px; border: 1px solid #cbd5e0; border-radius: 6px; font-size: 1em; }
        .btn-add  { background: #2b6cb0; color: white; border: none; border-radius: 6px;
                    padding: 9px 22px; font-size: 1em; cursor: pointer; }
        .btn-paid { background: #276749; color: white; border: none; border-radius: 6px;
                    padding: 10px 28px; font-size: 1.05em; cursor: pointer; margin-top: 16px; }
    </style>
</head>
<body>
    <h1>Order #{{ order.pk }} — Adding Items</h1>
    <p>Cashier: <strong>{{ order.cashier }}</strong></p>

    <!-- Current items table -->
    <table>
        <thead><tr><th>Product</th><th>Qty</th><th>Unit Price</th><th>Subtotal</th></tr></thead>
        <tbody>
        {% for item in items %}
        <tr>
            <td>{{ item.product.name }}</td>
            <td>{{ item.quantity }}</td>
            <td>${{ item.unit_price }}</td>
            <td>${{ item.subtotal }}</td>
        </tr>
        {% empty %}
        <tr><td colspan="4" style="color:#888;text-align:center">No items yet.</td></tr>
        {% endfor %}
        </tbody>
    </table>
    <p class="total">Order Total: ${{ order.total }}</p>

    <!-- Add item form -->
    <form class="add" method="post">
        {% csrf_token %}
        {% for field in item_form %}
        <div>
            <label>{{ field.label }}</label>
            {{ field }}
        </div>
        {% endfor %}
        <button class="btn-add" type="submit">＋ Add Item</button>
    </form>

    <!-- Mark Paid -->
    <form method="post">
        {% csrf_token %}
        <button class="btn-paid" name="mark_paid" type="submit">✔ Mark Order as Paid</button>
    </form>
</body>
</html>


### ដំណាក់កាល 4 — បន្ថែម URL Patterns ថ្មី

ធ្វើឲ្យ `sales/urls.py` រួមបញ្ចូល views ថ្មីទាំងពីរ៖


In [ ]:
# sales/urls.py  (updated — replace the whole file)

from django.urls import path
from . import views

urlpatterns = [
    # ── Part 1 views ──────────────────────────────────────
    path('products/',              views.product_list,   name='product_list'),
    path('products/<int:pk>/',     views.product_detail, name='product_detail'),
    path('orders/',                views.order_list,     name='order_list'),

    # ── Part 2 views (new) ────────────────────────────────
    path('orders/new/',            views.create_order,   name='create_order'),
    path('orders/<int:pk>/items/', views.add_item,       name='add_item'),
]

# Full URL examples:
# GET  /sales/orders/new/         → blank order form
# POST /sales/orders/new/         → creates order, redirects to add-items page
# GET  /sales/orders/3/items/     → add items to order #3
# POST /sales/orders/3/items/     → save a line item OR mark order as paid


---

## Authentication អ្នកប្រើ — Staff Login & Logout

Django មានប្រព័ន្ធ authentication ល្អរួមចំណែក — អ្នកមិនចាំបាច់សរសេរ login pages ពីសូន្យទេ។

### វា​ដំណើរការ​យ៉ាងដូចម្តេច

```
User visits /  (root URL)
        ↓  (RedirectView sends them to /accounts/login/)
Django shows the built-in login form
        ↓
Staff enter username + password
        ↓
Django checks credentials, creates a session
        ↓
Redirects to /sales/products/  (LOGIN_REDIRECT_URL)
        ↓
request.user is now the logged-in User object everywhere
```

### ដំណាក់កាល 1 — Wire Up Auth URLs និង Default Home Page ក្នុង `posdb/urls.py`

ពីររឿងត្រូវបន្ថែម៖
- `RedirectView` — បញ្ជូនអ្នកណាមួយដែលចូល `/` ទៅកាន់ login page
- `django.contrib.auth.urls` — ផ្តល់ login, logout, និង password views ដោយឥតគិតថ្លៃ


In [ ]:
# posdb/urls.py  (updated)

from django.contrib import admin
from django.urls import path, include
from django.views.generic import RedirectView

urlpatterns = [
    path('',          RedirectView.as_view(url='/accounts/login/'), name='home'),  # / → login page"
    path('admin/',    admin.site.urls),
    path('sales/',    include('sales.urls')),
    path('accounts/', include('django.contrib.auth.urls')),  # login, logout, password change
]

# URL map:
# /                  → redirect to /accounts/login/
# /accounts/login/   → login page
# /accounts/logout/  → logs the user out
# /sales/products/   → product catalogue  (requires login)
# /sales/orders/     → orders list        (requires login)
# /admin/            → Django admin panel


### ដំណាក់កាល 2 — បង្កើត Login Template

Django auth views នឹងស្វែងរក template មានឈ្មោះ `registration/login.html`។

បង្កើត folder `templates/registration/` នៅ **root project** និងបន្ថែម `login.html`:

> សូមធានាថា `posdb/settings.py` មាន `TEMPLATES` ដែលចាត់ទុក `BASE_DIR / 'templates'` ក្នុង `'DIRS'`:
> ```python
> TEMPLATES = [{ 'DIRS': [BASE_DIR / 'templates'], ... }]
> ```


In [ ]:
<!-- templates/registration/login.html -->
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Staff Login — POS</title>
    <style>
        body  { font-family: Arial, sans-serif; display: flex; justify-content: center;
                align-items: center; min-height: 100vh; background: #edf2f7; margin: 0; }
        .box  { background: white; padding: 40px 48px; border-radius: 12px;
                box-shadow: 0 4px 16px rgba(0,0,0,0.1); width: 340px; }
        h1    { color: #1a202c; margin-bottom: 24px; text-align: center; }
        label { font-weight: bold; display: block; margin-top: 14px; color: #2d3748; }
        input { width: 100%; padding: 9px 11px; margin-top: 6px; border: 1px solid #cbd5e0;
                border-radius: 6px; box-sizing: border-box; font-size: 1em; }
        button { margin-top: 24px; width: 100%; padding: 11px; background: #2b6cb0;
                 color: white; border: none; border-radius: 7px; font-size: 1.05em; cursor: pointer; }
        button:hover { background: #2c5282; }
        .errors { color: #c53030; font-size: 0.88em; margin-top: 10px; }
    </style>
</head>
<body>
    <div class="box">
        <h1>🛒 POS Staff Login</h1>
        <form method="post">
            {% csrf_token %}
            {{ form.as_p }}
            {% if form.errors %}
            <p class="errors">Invalid username or password. Please try again.</p>
            {% endif %}
            <button type="submit">Log In</button>
        </form>
    </div>
</body>
</html>


### ដំណាក់កាល 3 — កំណត់ Redirect URLs ក្នុង `settings.py`

បន្ទាប់ពី login/logout, Django នឹង redirect user ទៅទីកន្លែងមួយ។ ប្រាប់វាថាត្រូវ redirect ទៅណា៖


In [ ]:
# posdb/settings.py  — add these two lines anywhere at the bottom

LOGIN_REDIRECT_URL  = '/sales/products/'   # after login → product catalogue
LOGOUT_REDIRECT_URL = '/accounts/login/'   # after logout → back to login page

# With these settings + the root RedirectView, the full navigation cycle is:
#   /  →  /accounts/login/  →  (log in)  →  /sales/products/
#   (log out)  →  /accounts/login/


## ការពារ Views ដោយ `@login_required`

បន្ថែម `@login_required` លើ view function នីមួយៗ នឹង **រារាំងអ្នកមិនបាន authenticate**។ ប្រសិនបើអ្នកមិនបាន login ពេលព្យាយាមចូល URL នោះ Django នឹង redirect ទៅ `/accounts/login/`។


In [ ]:
# sales/views.py  — complete file with authentication applied to all views

from django.shortcuts import render, get_object_or_404, redirect
from django.contrib.auth.decorators import login_required
from .models import Product, Order, OrderItem
from .forms import OrderItemForm


@login_required
def product_list(request):
    products = Product.objects.filter(is_active=True)
    return render(request, 'sales/product_list.html', {'products': products})


@login_required
def product_detail(request, pk):
    product = get_object_or_404(Product, pk=pk)
    return render(request, 'sales/product_detail.html', {'product': product})


@login_required
def order_list(request):
    orders = Order.objects.all()
    return render(request, 'sales/order_list.html', {'orders': orders})


@login_required
def create_order(request):
    """Instantly create an open order and jump straight to the add-items page."""
    order = Order.objects.create(
        cashier=request.user,
        status='open',
    )
    return redirect('add_item', pk=order.pk)


@login_required
def add_item(request, pk):
    order = get_object_or_404(Order, pk=pk)
    if request.method == 'POST':
        if 'mark_paid' in request.POST:
            order.status = 'paid'
            order.save()
            return redirect('order_list')
        item_form = OrderItemForm(request.POST)
        if item_form.is_valid():
            item            = item_form.save(commit=False)
            item.order      = order
            item.unit_price = item.product.price
            item.save()
            return redirect('add_item', pk=order.pk)
    else:
        item_form = OrderItemForm()
    return render(request, 'sales/add_item.html', {
        'order': order,
        'item_form': item_form,
        'items': order.items.select_related('product'),
    })


### បន្ថែម Link ចេញពី session (Logout) ទៅក្នុង Templates របស់អ្នក

គ្រប់ទំព័រត្រូវមានវិធីសម្រាប់ staff ចេញពីប្រព័ន្ធ។ បន្ថែម snippet នេះទៅ navigation bar ក្នុង `product_list.html` និង `order_list.html`:


In [ ]:
<!-- Add this <nav> block to the top of product_list.html and order_list.html -->
<nav>
    <a href="/sales/products/">Products</a>
    <a href="/sales/orders/">Orders</a>
    <a href="/sales/orders/new/">＋ New Order</a>
    <a href="/admin/">Admin</a>
    <span style="float:right">
        👤 {{ request.user.username }}
        <form method="post" action="/accounts/logout/" style="display:inline">
            {% csrf_token %}
            <button type="submit" style="background:none;border:none;color:#2b6cb0;cursor:pointer;padding:0;font-size:1em">
                Log Out
            </button>
        </form>
    </span>
</nav>


---

## ភ្ជាប់ Orders ទៅ Cashier ដែលបាន login

បច្ចុប្បន្ន `cashier` ជា `CharField` តែប៉ុណ្ណោះ។ វិធីល្អជាងគេគឺប្រើ **ForeignKey ទៅ model `User`** ដូច្នេះ order នីមួយៗភ្ជាប់ទៅ account staff ពិតប្រាកដ។

### កែប្រែ `Order` Model


In [ ]:
# sales/models.py  — update the Order model

from django.db import models
from django.contrib.auth.models import User   # ← import Django's built-in User


class Order(models.Model):
    STATUS_CHOICES = [
        ('open',      'Open'),
        ('paid',      'Paid'),
        ('refunded',  'Refunded'),
        ('cancelled', 'Cancelled'),
    ]

    # Replace 'cashier = models.CharField(...)' with a ForeignKey:
    cashier    = models.ForeignKey(
                    User,
                    on_delete=models.SET_NULL,   # keep the order if the staff account is deleted
                    null=True,
                    related_name='orders',
                 )
    status     = models.CharField(max_length=20, choices=STATUS_CHOICES, default='open')
    created_at = models.DateTimeField(auto_now_add=True)
    notes      = models.TextField(blank=True)

    @property
    def total(self):
        return sum(item.subtotal for item in self.items.all())

    def __str__(self):
        name = self.cashier.username if self.cashier else 'unknown'
        return f"Order #{self.pk}  [{self.status.upper()}]  by {name}  —  ${self.total:.2f}"

    class Meta:
        ordering = ['-created_at']


បន្ទាប់ពីផ្លាស់ប្តូរម៉ូដែល run migrations ៖

```bash
python manage.py makemigrations sales
python manage.py migrate
```

បន្ទាប់មកធ្វើឲ្យ `create_order` ក្នុង `views.py` assign `request.user` ត្រូវការ object `User` ទៅ `order.cashier` ដោយផ្ទាល់៖

```python
order.cashier = request.user   # ← assign the User object, not a string
```

### សង្ខេប Authentication

| Feature | វា​ដំណើរការ​យ៉ាងដូចម្តេច |
|---|---|
| `include('django.contrib.auth.urls')` | បន្ថែម login, logout, password-change URLs ឲ្យដោយស្វ័យ |
| `@login_required` | រារាំង anonymous users ពី view |
| `request.user` | Object `User` ដែលបាន login (ឬ `AnonymousUser`) |
| `request.user.is_authenticated` | `True` ប្រសិនបើមានអ្នក login |
| `LOGIN_REDIRECT_URL` | ទីតាំង redirect បន្ទាប់ពី login ជោគជ័យ |
| `LOGOUT_REDIRECT_URL` | ទីតាំង redirect បន្ទាប់ពី logout |
| **CSRF token** | `{% csrf_token %}` — ត្រូវនៅក្នុង POST form រាល់សំណុំ, គឺការការពារការវាយប្រហារ CSRF |

---


---

## មូលដ្ឋានសម្រាប់ Deployment

ពេលដែលអ្នកត្រៀមរួចដាក់ POS លើម៉ាស៊ីនពិត មានបីរឿងដែលត្រូវផ្លាស់ពីការអភិវឌ្ឍន៍

### 1 — បិទ Debug Mode និងការការពារ Secret Key

នៅការអភិវឌ្ឍ `DEBUG = True` បង្ហាញ error page ពហុព័ត៌មាន — ជាតុល្យសម្រាប់ production។ `SECRET_KEY` មិនត្រូវ commit ទៅ version control។

ប្រើ **environment variables** ដើម្បីរក្សាឲ្យវាខ្វះពីកូដ៖


In [ ]:
pip install python-dotenv


បង្កើត `.env` នៅ root project (ចាំបញ្ជាក់បន្ថែម `.env` ទៅ `.gitignore`!) :

```
SECRET_KEY=your-very-long-random-secret-key-here
DEBUG=False
ALLOWED_HOSTS=yourdomain.com,www.yourdomain.com
```

បន្ទាប់មកផ្លាស់ប្ដូរ `posdb/settings.py`៖


In [ ]:
# posdb/settings.py  — production-ready top section

import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()   # reads .env file into environment variables

BASE_DIR = Path(__file__).resolve().parent.parent

SECRET_KEY = os.environ['SECRET_KEY']                          # required — crash if missing
DEBUG      = os.getenv('DEBUG', 'False') == 'True'            # False by default
ALLOWED_HOSTS = os.getenv('ALLOWED_HOSTS', '127.0.0.1').split(',')

# ... rest of settings.py stays the same


### 2 — ប្រមូល static files

នៅ production វេបសើវ័រ (មិនមែន Django) នឹងបម្រើ CSS/JS។ រត់បញ្ចប់នេះមួយដងមុន deploy៖

```bash
python manage.py collectstatic
```

បន្ថែមក្នុង `settings.py` :
```python
STATIC_ROOT = BASE_DIR / 'staticfiles'
```

### 3 — ប្រើ Production Server (`gunicorn`)

Django's built-in `runserver` សុទ្ធសម្រាប់ development តែប៉ុណ្ណោះ។ ត្រូវដំឡើង **gunicorn** សម្រាប់ production៖


In [ ]:
pip install gunicorn

# Start the production server (from the project root folder):
# gunicorn posdb.wsgi:application --bind 0.0.0.0:8000 --workers 3


### Deployment Checklist

| Step | Command / Setting |
|---|---|
| Secret key from env | `SECRET_KEY = os.environ['SECRET_KEY']` |
| Debug off | `DEBUG = False` |
| Set allowed hosts | `ALLOWED_HOSTS = ['yourdomain.com']` |
| Collect static files | `python manage.py collectstatic` |
| Run migrations | `python manage.py migrate` |
| Use gunicorn | `gunicorn posdb.wsgi:application` |
| HTTPS | Use a reverse proxy like **nginx** + **Let's Encrypt** certificate |

---


---

## សេចក្តីសន្និដ្ឋាន (Conclusion)

### អ្វីដែលអ្នកបានរៀននៅ ផ្នែក 2

| Concept | វាធ្វើអ្វី |
|---|---|
| **`ModelForm`** | បង្កើត form ពី Model ដោយស្វ័យ — មិនចាំបាច់ចម្លង field |
| **`is_valid()` / `cleaned_data`** | ផ្ទៀងផ្ទាត់ POST data និងបង្ហាញតម្លៃ Python ស្អាត |
| **`commit=False`**  | អនុញ្ញាតកែប្រែ instance មុនរក្សាទុក |
| **`Order.objects.create()`** | បង្កើត record បន្ទាន់ ដោយគ្មាន form — ប្រើសម្រាប់ order ខ្លួនឯង |
| **`@login_required`** | ការការពារ១បន្ទាត់ — redirect anonymous users ទៅ login page |
| **`request.user`** | Object `User` ដែលបាន login, មាននៅក្នុង view ដែលបានការពារ |
| **`ForeignKey(User)`** | ភ្ជាប់ `Order` ទៅ account staff ពិតប្រាកដ |
| **`RedirectView`** | view ដើម្បី redirect URL មួយទៅ URL មួយទៀត — ប្រើសម្រាប់ homepage |
| **`include('django.contrib.auth.urls')`** | ផ្ដល់ login/logout/password views ដោយម្តង |
| **`DEBUG = False`** | ត្រូវនៅ production — លាក់ព័ត៌មានលំអិតពីកំហុសពីសាធារណៈ |
| **environment variables** | រក្សា secrets ចេញពី source code |
| **gunicorn** | WSGI server សម្រាប់ production ដើម្បីជំនួស `runserver` |

### រចនាសម្ព័ន្ធឯកសារ Part 2 ពេញលេញ

```
posdb/
├── .env                       ← secrets (never commit this!)
├── manage.py
├── templates/
│   └── registration/
│       └── login.html         ← staff login page
├── posdb/
│   ├── settings.py            ← LOGIN_REDIRECT_URL, LOGOUT_REDIRECT_URL, STATIC_ROOT
│   └── urls.py                ← root redirect + django.contrib.auth.urls
└── sales/
    ├── forms.py               ← OrderItemForm only
    ├── models.py              ← Order.cashier is ForeignKey(User)
    ├── views.py               ← all views protected with @login_required
    ├── urls.py                ← URL patterns
    └── templates/sales/
        ├── product_list.html
        ├── order_list.html
        ├── product_detail.html
        └── add_item.html
```

### ផែនទី URL ពេញលេញ

| URL | វាធ្វើអ្វី |
|---|---|
| `/` | Redirected to `/accounts/login/` |
| `/accounts/login/` | Staff login page |
| `/accounts/logout/` | Logs out, redirects to `/accounts/login/` |
| `/sales/products/` | Product catalogue (login required) |
| `/sales/products/<id>/` | Product detail page (login required) |
| `/sales/orders/` | All orders (login required) |
| `/sales/orders/new/` | Create order + go to add-items (login required) |
| `/sales/orders/<id>/items/` | Add items / mark paid (login required) |
| `/admin/` | Django admin panel |

### Workflow នៃ Order (ផ្នែក 2)

```
Visit /  →  redirected to /accounts/login/
        ↓  (log in)
/sales/products/  — browse products
        ↓  click "＋ New Order"
create_order view  →  order created instantly
        ↓
/sales/orders/<pk>/items/  — add products, set quantities
        ↓  click "✔ Mark Order as Paid"
/sales/orders/  — see all orders with totals
```

### អ្វីខាងមុខសប្តាហ៍ក្រោយ

- **Class-Based Views** — `ListView`, `CreateView`, `UpdateView` ជាជម្រើសដ៏ខ្លាំងសម្រាប់ function views
- **REST API** — Django REST Framework ដើម្បីផ្ដល់ទិន្នន័យជា JSON សម្រាប់ mobile app
- **Background Tasks** — ផ្ញើអ៊ីម៉ែល receipt ជាការងារ asynchronous ជាមួយ Celery

---


---

## ការផ្តល់ការងារផ្ទះ (Homework) 🏠

បំពេញភារកិច្ចខាងក្រោម ដើម្បីរឹតបន្តឹងអ្វីដែលបានរៀននៅ ផ្នែក 2។ រាល់ភារកិច្ចស្ថិតលើកូដពីមេរៀននេះ។

---

### ភារកិច្ច 1 — ព្យាយាម Workflow ពេញលេញ (ងាយ)

1. ប្រើ server: `python manage.py runserver`
2. ទៅវើ `http://127.0.0.1:8000/sales/products/` — អ្នកគួរត្រូវបាន redirect ទៅ login page។
3. ចូលជាមួយ superuser account។
4. បង្កើត order ថ្មី តាម `http://127.0.0.1:8000/sales/orders/new/`
5. បន្ថែមយ៉ាងហោចណាស់ **3 line items** និង mark order ជា **Paid**។
6. ធ្វើតេស្តថា order បង្ហាញនៅ orders list មាន total ត្រឹមត្រូវ។

---

### ភារៈកិច្ច 2 — Form Validation (ងាយ–មធ្យម)

ក្នុង `sales/forms.py`, បន្ថែម validation ការផ្ទៀងផ្ទាល់ទៅ `OrderItemForm`:
- ប្រសិនបើ product ដែលជ្រើស `stock` = `0`, បោះត error `ValidationError` ជាមួយសារ `"This product is out of stock."`

```python
def clean(self):
    cleaned = super().clean()
    product = cleaned.get('product')
    if product and product.stock == 0:
        raise forms.ValidationError("This product is out of stock.")
    return cleaned
```

សាកល្បងដោយកំណត់ product stock = 0 ក្នុង shell និងព្យាយាមបញ្ចូលវាទៅ order។

---

### ភារកិច្ច 3 — ទំព័រ "My Orders" (មធ្យម)

បន្ថែម view ថ្មី `/sales/orders/mine/` ដែលបង្ហាញតែ orders ដែលបង្កើតដោយ user ដែល login នៅពេលនេះ៖

```python
@login_required
def my_orders(request):
    orders = Order.objects.filter(cashier=request.user)
    return render(request, 'sales/order_list.html', {'orders': orders})
```

1. បន្ថែម URL pattern ទៅ `sales/urls.py`.
2. បន្ថែម link "My Orders" ទៅ navigation bar នៅ templates របស់អ្នក.

---

### ភារកិច្ច 4 — កាត់ស្តុក (Stock Deduction) (មធ្យម–យ៉ាប់កល់)

នៅពេល `OrderItem` ត្រូវបានរក្សា (បន្ទាប់ពី `item_form.save()`), បន្ថែម logic ដើម្បីកាត់ស្តុក product ដោយ `quantity`៖

Update `add_item` view in `views.py` :
```python
if item_form.is_valid():
    item            = item_form.save(commit=False)
    item.order      = order
    item.unit_price = item.product.price
    item.save()

    # Deduct stock
    product = item.product
    product.stock -= item.quantity
    product.save()

    return redirect('add_item', pk=order.pk)
```

សាកល្បងដោយបញ្ចូល items ហើយពិនិត្យថាស្តុក product ថយចុះនៅក្នុង admin panel។

---

### ភារកិច្ច 5 — ត្រៀមសម្រាប់ Deployment (មធ្យម)

1. បង្កើត `.env` ជាមួយ `SECRET_KEY`, `DEBUG=False`, និង `ALLOWED_HOSTS=127.0.0.1`.
2. ផ្លាស់ប្ដូរ `posdb/settings.py` ដើម្បីអានពី `.env` ដោយប្រើ `python-dotenv`.
3. រត់ `python manage.py check --deploy` និងចុះសេចក្តីប្រកាស warnings ដែលវាបង្ហាញ។

---

### ភារកិច្ច 6 — Bonus Challenge ⭐

បន្ថែម **Cancel Order** button នៅ `add_item.html` ដែល៖
1. កំណត់ `order.status = 'cancelled'` និង save។
2. Redirect ទៅ order list។
3. ធ្វើឲ្ា_ORDER ដែល(cancelled) មិនអាចបន្ថែម items បានទៀត (ត្រួតពិនិត្យក្នុង `add_item` view និងបង្ហាញ error message ប្រសិនបើ order មិនមែន `open`)។

---

### កន្លែងឆ្លើយ

**Task 2 — តើ validation របស់អ្នកដំណើរការ? អត្ថាធិប្បាយអ្វីដែលកើតឡើង:**

**Task 3 — ចម្លង URL pattern ថ្មីរបស់អ្នកនៅទីនេះ:**
```python
# 
```

**Task 5 — `manage.py check --deploy` warnings:**
1. 
2. 
3. 

**Task 6 — Cancel button code (views.py logic):**
```python
# 
```

**Notes / Questions:**
